Copy of Phase 3 for Silver folder:

In [ ]:

# Distributed Environment Setup
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz
!tar xf spark-3.5.4-bin-hadoop3.tgz

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.4-bin-hadoop3"

In [ ]:
!pip install -q findspark pyspark==4.0.0

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F



from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("MovieLens_Phase3") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(" Spark Session initialized successfully!")
spark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
 Spark Session initialized successfully!


In [ ]:
# Data Ingestion
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Strict Schema Enforcement + Data Ingestion

In [ ]:
# Schemas
schema_movies = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), False),
    StructField("genres", StringType(), False)
])

schema_ratings = StructType([
    StructField("userId", LongType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", DoubleType(), False),
    StructField("timestamp", LongType(), False)
])

schema_tags = StructType([
    StructField("userId", LongType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), False)
])

schema_links = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("imdbId", LongType(), True),
    StructField("tmdbId", LongType(), True)
])

# Ingestion
#change bronze path to match your folder!

bronze_path="/content/drive/MyDrive/bronze"
df_movies_raw = spark.read.csv(bronze_path+"/movies.csv", header=True, schema=schema_movies)
df_ratings_raw = spark.read.csv(bronze_path+"/ratings.csv", header=True, schema=schema_ratings)
df_tags_raw = spark.read.csv(bronze_path+"/tags.csv", header=True, schema=schema_tags)
df_links_raw = spark.read.csv(bronze_path+"/links.csv", header=True, schema=schema_links)

print("Raw schemas loaded:")
df_movies_raw.printSchema()

Raw schemas loaded:
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [ ]:
# Type Conversion and Safe Casting
year_extract = regexp_extract(col("title"), r"\((\d{4})\)$", 1)
df_movies_clean = df_movies_raw \
    .withColumn("release_year", when(year_extract != "", year_extract.cast("int")).otherwise(lit(None).cast("int"))) \
    .withColumn("title", regexp_replace(col("title"), r" \(\d{4}\)$", ""))

In [ ]:
# Timestamp conversion
df_ratings_clean = df_ratings_raw \
    .withColumn("rating_timestamp", from_unixtime(col("timestamp")).cast("timestamp")) \
    .drop("timestamp")

df_tags_clean = df_tags_raw \
    .withColumn("tag_timestamp", from_unixtime(col("timestamp")).cast("timestamp")) \
    .drop("timestamp")

In [ ]:
# Safe casting example for links
df_links_clean = df_links_raw \
    .withColumn("imdbId", when(col("imdbId").cast("string").rlike("^[0-9]+$"), col("imdbId").cast("long")).otherwise(lit(None))) \
    .withColumn("tmdbId", when(col("tmdbId").cast("string").rlike("^[0-9]+$"), col("tmdbId").cast("long")).otherwise(lit(None)))

In [ ]:
# Data Enrichment – NEW calculated column (rating category)
df_ratings_clean = df_ratings_clean.withColumn("rating_category",
    when(col("rating") >= 4.5, "Excellent") \
    .when(col("rating") >= 4.0, "Very Good") \
    .when(col("rating") >= 3.0, "Good") \
    .otherwise("Average")
)

In [ ]:
# Structural Cleaning
df_movies_exploded = df_movies_clean \
    .withColumn("genre", explode(split(col("genres"), "\\|")))

In [ ]:
def set_column_nullable(df, column_name, nullable=False):
    """Helper function to change nullable flag in schema"""
    new_schema = StructType([
        StructField(field.name, field.dataType, nullable if field.name == column_name else field.nullable)
        for field in df.schema.fields
    ])
    return spark.createDataFrame(df.rdd, new_schema)

In [ ]:
df_movies_clean = set_column_nullable(df_movies_clean, "movieId", nullable=False)
df_movies_clean = set_column_nullable(df_movies_clean, "title", nullable=False)
df_movies_clean = set_column_nullable(df_movies_clean, "genres", nullable=False)

df_ratings_clean = set_column_nullable(df_ratings_clean, "userId", nullable=False)
df_ratings_clean = set_column_nullable(df_ratings_clean, "movieId", nullable=False)
df_ratings_clean = set_column_nullable(df_ratings_clean, "rating", nullable=False)
df_ratings_clean = set_column_nullable(df_ratings_clean, "rating_timestamp", nullable=False)

df_links_clean = set_column_nullable(df_links_clean, "movieId", nullable=False)

In [ ]:
# SCHEMAS
print(" Schemas (string → int/double/timestamp + new columns):")
print("\n--- Movies Clean (movieId, title, genres now nullable=False) ---")
df_movies_clean.printSchema()

print("\n--- Ratings Clean (userId, movieId, rating, rating_timestamp now nullable=False) ---")
df_ratings_clean.printSchema()

print("\n--- Movies Exploded ---")
df_movies_exploded.printSchema()

print("\nPreview of exploded genres:")
df_movies_exploded.select("movieId", "title", "genre").show(5, truncate=False)

 Schemas (string → int/double/timestamp + new columns):

--- Movies Clean (movieId, title, genres now nullable=False) ---
root
 |-- movieId: integer (nullable = false)
 |-- title: string (nullable = false)
 |-- genres: string (nullable = false)
 |-- release_year: integer (nullable = true)


--- Ratings Clean (userId, movieId, rating, rating_timestamp now nullable=False) ---
root
 |-- userId: long (nullable = false)
 |-- movieId: integer (nullable = false)
 |-- rating: double (nullable = false)
 |-- rating_timestamp: timestamp (nullable = false)
 |-- rating_category: string (nullable = false)


--- Movies Exploded ---
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- genre: string (nullable = false)


Preview of exploded genres:
+-------+---------+---------+
|movieId|title    |genre    |
+-------+---------+---------+
|1      |Toy Story|Adventure|
|1      |Toy Story|Ani

In [ ]:
# Register temp views
df_movies_clean.createOrReplaceTempView("movies")
df_ratings_clean.createOrReplaceTempView("ratings")
df_tags_clean.createOrReplaceTempView("tags")
df_links_clean.createOrReplaceTempView("links")
df_movies_exploded.createOrReplaceTempView("movies_exploded")

# Print info proving that this worked
print("Registered temp views:")
spark.catalog.listTables()

print("\nmovies preview:")
spark.sql("SELECT * FROM movies LIMIT 5").show(truncate=False)

print("\nratings preview:")
spark.sql("SELECT * FROM ratings LIMIT 5").show(truncate=False)

print("\ntags preview:")
spark.sql("SELECT * FROM tags LIMIT 5").show(truncate=False)

print("\nlinks preview:")
spark.sql("SELECT * FROM links LIMIT 5").show(truncate=False)

print("\nmovies_exploded preview:")
spark.sql("SELECT * FROM movies_exploded LIMIT 5").show(truncate=False)

Registered temp views:

movies preview:
+-------+---------------------------+-------------------------------------------+------------+
|movieId|title                      |genres                                     |release_year|
+-------+---------------------------+-------------------------------------------+------------+
|1      |Toy Story                  |Adventure|Animation|Children|Comedy|Fantasy|1995        |
|2      |Jumanji                    |Adventure|Children|Fantasy                 |1995        |
|3      |Grumpier Old Men           |Comedy|Romance                             |1995        |
|4      |Waiting to Exhale          |Comedy|Drama|Romance                       |1995        |
|5      |Father of the Bride Part II|Comedy                                     |1995        |
+-------+---------------------------+-------------------------------------------+------------+


ratings preview:
+------+-------+------+-------------------+---------------+
|userId|movieId|rating|rat

In [ ]:
#  Filter
high_ratings = df_ratings_clean.filter(col("rating") >= 4.0)

In [ ]:
# Join
joined = high_ratings.join(df_movies_clean, "movieId", "inner")

In [ ]:
# GroupBy + Agg
agg_ratings = joined.groupBy("title") \
    .agg(count("*").alias("rating_count"), avg("rating").alias("avg_rating"))

In [ ]:
# Additional filter + sort
top_movies = agg_ratings.filter(col("rating_count") >= 1000) \
    .orderBy(col("avg_rating").desc(), col("rating_count").desc()) \
    .limit(20)

In [ ]:
# Genre explosion + aggregation (structural)
genre_stats = df_movies_exploded.join(df_ratings_clean, "movieId") \
    .groupBy("genre") \
    .agg(count("*").alias("rating_events"), round(avg("rating"), 3).alias("avg_rating")) \
    .orderBy(col("rating_events").desc())

In [ ]:
# Join with tags + KPI
tagged_high = df_tags_clean.join(df_ratings_clean, ["userId", "movieId"]) \
    .filter(col("rating") >= 4.0) \
    .groupBy("tag") \
    .agg(count("*").alias("tag_uses"), round(avg("rating"), 3).alias("avg_rating")) \
    .filter(col("tag_uses") >= 50) \
    .orderBy(col("avg_rating").desc())

print("Top 20 highest-rated movies (DataFrame API):")
top_movies.show(5)

Top 20 highest-rated movies (DataFrame API):
+--------------------+------------+------------------+
|               title|rating_count|        avg_rating|
+--------------------+------------+------------------+
|     Planet Earth II|        1741|4.6726019529006315|
|        Planet Earth|        2595| 4.663776493256262|
|    Band of Brothers|        2449| 4.658023683135974|
|      Godfather, The|       54746| 4.643480802250393|
|Shawshank Redempt...|       88753|4.6431050218020795|
+--------------------+------------+------------------+
only showing top 5 rows



Identical Analytics via Spark SQL

In [ ]:
# KPI via SQL
kpi_sql = """
SELECT
    m.title,
    COUNT(*) AS rating_count,
    ROUND(AVG(r.rating), 3) AS avg_rating
FROM ratings r
JOIN movies m ON r.movieId = m.movieId
GROUP BY m.title
HAVING COUNT(*) >= 1000
ORDER BY avg_rating DESC, rating_count DESC
LIMIT 20
"""
spark.sql(kpi_sql).show(5)

+--------------------+------------+----------+
|               title|rating_count|avg_rating|
+--------------------+------------+----------+
|     Planet Earth II|        1956|     4.447|
|        Planet Earth|        2952|     4.442|
|    Band of Brothers|        2811|     4.427|
|Shawshank Redempt...|      102929|     4.405|
|      Godfather, The|       66440|     4.317|
+--------------------+------------+----------+
only showing top 5 rows



In [ ]:
# Extra SQL query (genre-level KPI)
genre_ratings=spark.sql("""
SELECT
    genre,
    COUNT(*) AS rating_events,
    ROUND(AVG(rating), 3) AS avg_rating
FROM movies_exploded
JOIN ratings ON movies_exploded.movieId = ratings.movieId
GROUP BY genre
HAVING COUNT(*) >= 5000
ORDER BY  avg_rating DESC,rating_events DESC
LIMIT 10
""")

In [ ]:
# Define the target directory
silver_path = "/content/drive/MyDrive/silver"

# Primary outputs
df_movies_clean.write.mode("overwrite").parquet(f"{silver_path}/cleaned_movies_parquet")
df_ratings_clean.write.mode("overwrite").csv(f"{silver_path}/cleaned_ratings_csv", header=True)

# Optional optimization step
tagged_high.coalesce(1).write.mode("overwrite").csv(f"{silver_path}/tagged_high_csv", header=True)

df_movies_exploded.write.mode("overwrite").parquet(f"{silver_path}/movies_exploded_parquet")

print(f"Files written to: {silver_path}")

# Using a shell variable to list the specific directory
!ls -lh {silver_path}/cleaned_movies_parquet/ {silver_path}/cleaned_ratings_csv/ {silver_path}/tagged_high_csv/ | head -15

print("\nParquet example (columnar, efficient):")
spark.read.parquet(f"{silver_path}/cleaned_movies_parquet").printSchema()

Files written to: /content/drive/MyDrive/silver
/content/drive/MyDrive/silver/cleaned_movies_parquet/:
total 2.0M
-rw------- 1 root root 2.0M Apr 14 17:18 part-00000-991f898b-9ada-4c7b-a467-82356fb69718-c000.snappy.parquet
-rw------- 1 root root  14K Apr 14 17:18 part-00001-991f898b-9ada-4c7b-a467-82356fb69718-c000.snappy.parquet
-rw------- 1 root root    0 Apr 14 17:18 _SUCCESS

/content/drive/MyDrive/silver/cleaned_ratings_csv/:
total 1.5G
-rw------- 1 root root 231M Apr 14 17:36 part-00000-3e6775b9-5fc5-42e7-a81d-6b03c7d64843-c000.csv
-rw------- 1 root root 229M Apr 14 17:36 part-00001-3e6775b9-5fc5-42e7-a81d-6b03c7d64843-c000.csv
-rw------- 1 root root 229M Apr 14 17:55 part-00002-3e6775b9-5fc5-42e7-a81d-6b03c7d64843-c000.csv
-rw------- 1 root root 226M Apr 14 17:54 part-00003-3e6775b9-5fc5-42e7-a81d-6b03c7d64843-c000.csv
-rw------- 1 root root 226M Apr 14 18:14 part-00004-3e6775b9-5fc5-42e7-a81d-6b03c7d64843-c000.csv
-rw------- 1 root root 226M Apr 14 18:14 part-00005-3e6775b9-5fc

Gold folder contents

In [ ]:
# Remember to change path to make sure this actually writes!
gold_path = "/content/drive/MyDrive/gold"

df_top_twenty = top_movies.limit(20)
df_top_twenty.coalesce(1).write.mode("overwrite").parquet(f"{gold_path}/top_20_movies")

# Display the results in the console for immediate verification
df_top_twenty.show()

print(f"Top 20 rows successfully written to: {gold_path}/top_20_movies")

Py4JJavaError: An error occurred while calling o398.parquet.
: java.io.FileNotFoundException: File file:/content/drive/MyDrive/gold/top_20_movies/_temporary/0 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:597)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(FileOutputCommitter.java:334)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.commitJobInternal(FileOutputCommitter.java:404)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.commitJob(FileOutputCommitter.java:377)
	at org.apache.parquet.hadoop.ParquetOutputCommitt

In [ ]:
df_top_twenty_genres = genre_ratings.limit(20)
df_top_twenty_genres.coalesce(1).write.mode("overwrite").parquet(f"{gold_path}/top_20_genres")

# Display the results in the console for immediate verification
df_top_twenty_genres.show()

print(f"Top 20 rows successfully written to: {gold_path}/top_20_genres")

+---------+-------------+----------+
|    genre|rating_events|avg_rating|
+---------+-------------+----------+
|    Drama|     13973271|     3.682|
|   Comedy|     11206925|     3.432|
|   Action|      9665213|     3.476|
| Thriller|      8679464|     3.532|
|Adventure|      7590522|     3.523|
|   Sci-Fi|      5717337|     3.492|
|  Romance|      5524615|     3.545|
|    Crime|      5373051|     3.692|
|  Fantasy|      3702759|     3.512|
| Children|      2731841|     3.439|
+---------+-------------+----------+

Top 20 rows successfully written to: /content/drive/MyDrive/gold/top_20_genres
